# 02 — Prepare and validate the instruction dataset

This laptop-safe notebook validates all three splits and supports two line-level JSONL schemas:

1. `{'input': '...', 'output': '...'}`
2. `{'messages': [{'role': 'user', 'content': '...'}, {'role': 'assistant', 'content': '...'}]}`

The sample files deliberately mix the two schemas. Validation happens before transformation so file and line errors remain easy to diagnose.

In [1]:
from collections import Counter
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILES = [
    PROJECT_ROOT / 'data' / 'train.jsonl',
    PROJECT_ROOT / 'data' / 'validation.jsonl',
    PROJECT_ROOT / 'data' / 'test.jsonl',
]
print('Project root:', PROJECT_ROOT)

Project root: C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series


In [2]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'validate_dataset.py'),
    *map(str, DATA_FILES),
]
completed = subprocess.run(command, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
completed.check_returncode()

C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\data\train.jsonl: records=8, valid=8, invalid=0, input/output=4, messages=4, blank=0
C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\data\validation.jsonl: records=3, valid=3, invalid=0, input/output=2, messages=1, blank=0
C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\data\test.jsonl: records=3, valid=3, invalid=0, input/output=1, messages=2, blank=0

Validation passed: 14 valid record(s) across 3 file(s).



## Normalize records in memory

Training and serving libraries often prefer chat messages. The following helper converts `input`/`output` examples to user/assistant turns while leaving chat records unchanged. It does not modify source files.

In [3]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8-sig') as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'{path}:{line_number}: {exc}') from exc
    return records

def to_messages(record: dict) -> list[dict[str, str]]:
    if 'messages' in record:
        return [dict(message) for message in record['messages']]
    return [
        {'role': 'user', 'content': record['input'].strip()},
        {'role': 'assistant', 'content': record['output'].strip()},
    ]

splits = {path.stem: load_jsonl(path) for path in DATA_FILES}
normalized = {name: [to_messages(record) for record in records] for name, records in splits.items()}

for name, conversations in normalized.items():
    role_counts = Counter(message['role'] for conversation in conversations for message in conversation)
    print(f'{name:10} examples={len(conversations):2} roles={dict(role_counts)}')

train      examples= 8 roles={'user': 8, 'assistant': 8, 'system': 1}
validation examples= 3 roles={'user': 3, 'assistant': 3}
test       examples= 3 roles={'user': 3, 'assistant': 3}


In [4]:
preview = normalized['train'][0]
print(json.dumps(preview, indent=2, ensure_ascii=False))

assert all(conversation for split in normalized.values() for conversation in split)
assert all(
    any(message['role'] == 'user' for message in conversation)
    and any(message['role'] == 'assistant' for message in conversation)
    for split in normalized.values()
    for conversation in split
)
print('In-memory checks passed.')

[
  {
    "role": "user",
    "content": "What is parameter-efficient fine-tuning?"
  },
  {
    "role": "assistant",
    "content": "Parameter-efficient fine-tuning updates a small set of added or selected parameters while keeping most pretrained model weights frozen."
  }
]
In-memory checks passed.


In [5]:
COUNT_TOKENS = False  # Opt in: downloads tokenizer files from Hugging Face.
TOKENIZER_ID = 'Qwen/Qwen3-1.7B'

if COUNT_TOKENS:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, trust_remote_code=False)
    lengths = []
    for conversation in normalized['train']:
        token_ids = tokenizer.apply_chat_template(conversation, tokenize=True)
        lengths.append(len(token_ids))
    print('Train token lengths:', lengths)
    print('Maximum:', max(lengths))
else:
    print('Token counting skipped; validation does not require a model download.')

Token counting skipped; validation does not require a model download.


## Optional persistent conversion

The repository script converts `input`/`output` records and preserves existing chat records. It refuses to overwrite a file unless `--force` is passed. Generated files belong under the ignored `outputs/` directory.

In [6]:
RUN_CONVERSION = False
destination = PROJECT_ROOT / 'outputs' / 'train_messages.jsonl'
convert_command = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'convert_chat_jsonl.py'),
    str(PROJECT_ROOT / 'data' / 'train.jsonl'),
    str(destination),
]
print('Command:', ' '.join(convert_command))
if RUN_CONVERSION:
    subprocess.run(convert_command, check=True)
else:
    print('Persistent conversion skipped. Set RUN_CONVERSION=True to write the output.')

Command: C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\.venv\Scripts\python.exe C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\scripts\convert_chat_jsonl.py C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\data\train.jsonl C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\outputs\train_messages.jsonl
Persistent conversion skipped. Set RUN_CONVERSION=True to write the output.


## Before training

The included dataset is intentionally tiny. It is suitable for validating a pipeline and a two-step smoke test, not for producing a capable adapter. For a real experiment, add licensed, representative examples; remove duplicates and sensitive data; validate every split; and keep the test set untouched until final evaluation.